# Load and Inspect

In [1]:
# load data and inspect

import pandas as pd

df = pd.read_csv('delivered_orders.csv', parse_dates=['date_purchased'])
print(df.shape)
df.head()

/tmp/ipykernel_794/3261786910.py:5: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('delivered_orders.csv', parse_dates=['date_purchased'])


(96470, 14)


,order_id,customer_unique_id,customer_state,date_purchased,days_promised,actual_days,courier_days,number_of_days_late,order_is_late,seller_handed_over_late,number_of_items,item_cost,freight_cost,avg_review_score
0,eafa53108733896f6f18e2f8daca6f37,95700615deef776ed32faa08f0be634e,SP,2018-07-30,18,14,7.0,-4,False,False,1,159.90,15.43,5.0
1,29150127e6685892b6eab3eec79f59c7,290c77bc529b7ac935b93aa66c333dc3,SP,2018-01-12,25,17,14.0,-8,False,False,1,289.00,46.48,5.0
2,6b7d50bd145f6fc7f33cebabd7e49d0f,345ecd01c38d18a9036ed96c73b8d066,SP,2018-07-29,17,11,10.0,-6,False,False,1,230.00,22.25,5.0
3,5741ea1f91b5fbab2bd2dc653a5b5099,4c93744516667ad3b8f1fb645a3116a4,SC,2017-09-14,20,14,10.0,-6,False,False,1,259.90,22.31,5.0
4,36e694cf4cbc2a4803200c35e84abdc4,addec96d2e059c80c30fe6871d30d177,SP,2018-02-19,14,1,0.0,-13,False,False,1,14.99,7.78,5.0


# Check data types and missing values

In [2]:
df.info()
print()
print(df.isna().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 96470 entries, 0 to 96469
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   order_id                 96470 non-null  object        
 1   customer_unique_id       96470 non-null  object        
 2   customer_state           96470 non-null  object        
 3   date_purchased           96470 non-null  datetime64[ns]
 4   days_promised            96470 non-null  int64         
 5   actual_days              96470 non-null  int64         
 6   courier_days             96469 non-null  float64       
 7   number_of_days_late      96470 non-null  int64         
 8   order_is_late            96470 non-null  bool          
 9   seller_handed_over_late  96469 non-null  object        
 10  number_of_items          96470 non-null  int64         
 11  item_cost                96470 non-null  float64       
 12  freight_cost             96470 n

# Cleaningdrop 1 order with no courier hand-over date; fix the hand-over column’s data type.

In [3]:
# cleaning data

# dropping the one order with no courier hand-over date

df = df.dropna(subset=['courier_days']).copy()


# change the handover to distinct boolean values thus true/false

df['seller_handed_over_late'] = df['seller_handed_over_late'].astype(str).str.lower().map({'true': True, 'false': False})


print(df.shape)
print(df[['order_is_late','seller_handed_over_late']].dtypes)

(96469, 14)
order_is_late              bool
seller_handed_over_late    bool
dtype: object


# Sanity check for impossible values

In [4]:
# times for delivery should not be a negative number
print((df[['days_promised','actual_days','courier_days']]<0).sum())

df[['days_promised','actual_days','courier_days','number_of_days_late']].describe()

days_promised     0
actual_days       0
courier_days     20
dtype: int64


,days_promised,actual_days,courier_days,number_of_days_late
count,96469.000000,96469.000000,96469.000000,96469.000000
mean,24.372513,12.496439,9.282526,-11.876074
std,8.758197,9.554274,8.777037,10.181995
min,3.000000,0.000000,-16.000000,-147.000000
25%,19.000000,7.000000,4.000000,-17.000000
50%,24.000000,10.000000,7.000000,-12.000000
75%,29.000000,16.000000,12.000000,-7.000000
max,156.000000,210.000000,205.000000,188.000000


# Fixing bad courier values

In [5]:
df.loc[df['courier_days']<0,'courier_days'] = None


print((df['courier_days']<0).sum())

0


# Do late orders get worse reviews

In [12]:
from scipy.stats import mannwhitneyu

reviewed = df.dropna(subset=['avg_review_score'])
late = reviewed.loc[reviewed['order_is_late'], 'avg_review_score']
ontime = reviewed.loc[~reviewed['order_is_late'], 'avg_review_score']

stat,p = mannwhitneyu(late, ontime, alternative='less')
print(f"Late Orders: mean review {late.mean():.2f} (n={len(late)})")
print(f"Ontime Orders: mean review {ontime.mean():.2f} (n={len(ontime)})")
print(f"p-value:{p:.3g}")

Late Orders: mean review 2.27 (n=6380)
Ontime Orders: mean review 4.29 (n=89443)
p-value:0


>**Conclusion:** Late orders receive far worse reviews. Late orders average **2.27 stars** compared with **4.29 stars** for orders delivered on time or early (6,380 late vs 89,443 on-time reviewed orders).

>A Mann–Whitney U test confirms this difference is not due to chance (**p < 0.001**). I used Mann–Whitney rather than a t-test because review scores are 1–5 ratings, not continuous, normally distributed values.

>**Business impact:** Delivery reliability is a major driver of customer satisfaction. Reducing late deliveries should directly improve review scores, which affect seller reputation and buyer trust on the marketplace.

# Are the late orders reducing customer return rate?

In [17]:
from scipy.stats import chi2_contingency

# one row per the customers: was their order late, and did they ever order again?

customers = (df.sort_values('date_purchased')
  .groupby('customer_unique_id')
  .agg(first_order_late=('order_is_late','first'),
      total_orders=('order_id','count')))

customers['came_back'] = customers['total_orders'] > 1

table = pd.crosstab(customers['first_order_late'],customers['came_back'])
print(table)
print()
print(customers.groupby('first_order_late')['came_back'].mean().mul(100).round(2))

stat,p,dof,expected = chi2_contingency(table)

print(f"\np-value:{p:.3g}")

came_back         False  True 
first_order_late              
False             84354   2644
True               6194    157

first_order_late
False    3.04
True     2.47
Name: came_back, dtype: float64

p-value:0.0118


>Customers whose first order arrived late were less likely to place an order again. The effect is statistically significant but small, about 35 extra returning customers if late orders had matched the on-time rate. The main scost of late deliveries is bad reviews not lost repeat business. Rentention is low across the board (~97% never return), so it needs it own strategy.